# Exp04 — OOF blend of Exp00 and Exp03 Spatial NN

Reconstructed faithfully from the provided source files. **Run from the project directory** with `train.csv`, `test.csv`, `sample_submission.csv`, `cv_utils.py`, `wp_lib.py`, and the supplied model modules. The notebook does **not** claim to reproduce the original CSV without the original OOF files and the spatial model submission.

**Original script outputs only `id,mu`.** If the competition requires `id,mu,sigma`, this experiment alone is not a valid probabilistic submission. Do not invent sigma values.

## 1. Dependencies and project setup

Required packages: `numpy`, `pandas`, `scikit-learn`, `torch`. The original code uses the local `cv_utils.py`, `wp_lib.py`, `spatial_lib.py`, `exp03_lrdecay_cv.py`, and `exp04_oof_blend.py`. The three supplied scripts are included as text in the companion ZIP; the two other local modules were not provided.

In [5]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pathlib import Path
import os, json, sys, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
ROOT = Path.cwd()
print("Project directory:", ROOT)
required = ["train.csv", "test.csv", "sample_submission.csv", "cv_utils.py", "wp_lib.py", "spatial_lib.py"]
missing = [f for f in required if not (ROOT/f).exists()]
if missing: print("Missing project files:", missing)
else: print("Base dependencies available")

Project directory: /Users/secilnuragtepe/Desktop/wind-power-project/math482-wind-power-mean-2026
Base dependencies available


In [7]:
import torch
print(torch.__version__)

2.14.1


## 2. Experiment design and validation

For a blend weight `w`, the predicted mean is `w * spatial + (1-w) * exp00`. Candidate weights are 0, 0.05, ..., 1. The original experiment reports a fixed 50/50 blend, forward-chained weights (chosen on previous folds only), and optimistic in-sample weights (shown for reference, not unbiased validation). Final weights are fitted on all available OOF rows.

In [9]:
GRID = np.round(np.arange(0, 1.0001, 0.05), 2)
FOLDS = ["F1_2011Q2", "F2_2011Q3", "F3_2011Q4", "F4_2012Q1"]
def rmse_local(y, p):
    return float(np.sqrt(np.mean((np.asarray(y)-np.asarray(p))**2)))
def best_w(d, mask=None):
    q = d if mask is None else d[mask]
    if len(q)==0: raise ValueError("Empty weight-fitting group")
    return float(min(GRID, key=lambda w: rmse_local(q.power, w*q.sp + (1-w)*q.e0)))
def apply_blend(d, w_present, w_missing):
    w = np.where(d.weather_available == 1, w_present, w_missing)
    return w*d.sp + (1-w)*d.e0

## 3. Chronological OOF evaluation

Requires `oof/exp00_3seed_oof.csv` and either `oof/exp03_5seed_oof.csv` (preferred) or `oof/exp02_rescaled_3seed_oof.csv` (fallback). The selected spatial OOF file determines the matching spatial submission file.

In [11]:
oof_dir=ROOT/"oof"
e0_path=oof_dir/"exp00_3seed_oof.csv"
sp_path=oof_dir/"exp03_5seed_oof.csv"
if not sp_path.exists(): sp_path=oof_dir/"exp02_rescaled_3seed_oof.csv"
sp_sub=ROOT/("exp03_lrdecay_submission.csv" if sp_path.name.startswith("exp03") else "exp02_spatial_submission.csv")
print("OOF:",e0_path,sp_path,"\nSpatial submission:",sp_sub)
print("Available:",e0_path.exists(),sp_path.exists(),sp_sub.exists())

OOF: /Users/secilnuragtepe/Desktop/wind-power-project/math482-wind-power-mean-2026/oof/exp00_3seed_oof.csv /Users/secilnuragtepe/Desktop/wind-power-project/math482-wind-power-mean-2026/oof/exp03_5seed_oof.csv 
Spatial submission: /Users/secilnuragtepe/Desktop/wind-power-project/math482-wind-power-mean-2026/exp03_lrdecay_submission.csv
Available: True True True


In [12]:
if e0_path.exists() and sp_path.exists():
    e0=pd.read_csv(e0_path)
    sp=pd.read_csv(sp_path)[["id","pred"]].rename(columns={"pred":"sp"})
    d=e0.rename(columns={"pred":"e0"}).merge(sp,on="id",validate="one_to_one")
    assert set(["id","fold","power","weather_available","e0","sp"]).issubset(d.columns)
    results={"Exp00":d.e0, "Spatial":d.sp, "Fixed 50/50":.5*d.e0+.5*d.sp}
    fc_g=pd.Series(np.nan,index=d.index); fc_s=pd.Series(np.nan,index=d.index); logs=[]
    for k,f in enumerate(FOLDS):
        past=d[d.fold.isin(FOLDS[:k])]; current=d.fold.eq(f)
        if len(past):
            wg=best_w(past); wp=best_w(past,past.weather_available.eq(1)); wm=best_w(past,past.weather_available.eq(0))
        else: wg=wp=wm=.5
        fc_g.loc[current]=apply_blend(d.loc[current],wg,wg)
        fc_s.loc[current]=apply_blend(d.loc[current],wp,wm)
        logs.append(dict(fold=f,w_global=wg,w_present=wp,w_missing=wm))
    results["Forward chained global"]=fc_g
    results["Forward chained present/missing"]=fc_s
    wg=best_w(d); wp=best_w(d,d.weather_available.eq(1)); wm=best_w(d,d.weather_available.eq(0))
    results[f"In-sample global w={wg} (optimistic)"]=apply_blend(d,wg,wg)
    results[f"In-sample present={wp} missing={wm} (optimistic)"]=apply_blend(d,wp,wm)
    rows=[]
    for label,p in results.items():
        row={"method":label}
        for f in FOLDS:
            mask=d.fold.eq(f); row[f]=rmse_local(d.power[mask],p[mask])
        row["mean_CV"]=np.mean([row[f] for f in FOLDS]);row["pooled"]=rmse_local(d.power,p)
        row["missing"]=rmse_local(d.power[d.weather_available.eq(0)],p[d.weather_available.eq(0)])
        rows.append(row)
    display(pd.DataFrame(logs).set_index("fold"))
    display(pd.DataFrame(rows).set_index("method").round(5))
    oof_dir.mkdir(exist_ok=True)
    weights_path=oof_dir/"exp04_blend_weights.json"
    if weights_path.exists():
        print("Existing weights file preserved:",weights_path)
        print("Computed candidate:",dict(spatial_oof=str(sp_path.relative_to(ROOT)),w_global=wg,w_present=wp,w_missing=wm))
    else:
        weights_path.write_text(json.dumps(dict(spatial_oof=str(sp_path.relative_to(ROOT)),w_global=wg,w_present=wp,w_missing=wm),indent=2))
        print("Saved",weights_path)
else:
    print("OOF files missing; cannot calculate original weights. Copy the project OOF/ files here or run the original CV scripts.")

,w_global,w_present,w_missing
fold,,,
F1_2011Q2,0.50,0.5,0.5
F2_2011Q3,0.55,0.6,0.5
F3_2011Q4,0.55,0.6,0.5
F4_2012Q1,0.60,0.6,0.6


,F1_2011Q2,F2_2011Q3,F3_2011Q4,F4_2012Q1,mean_CV,pooled,missing
method,,,,,,,
Exp00,0.16889,0.13128,0.19001,0.21415,0.17608,0.17857,0.20376
Spatial,0.16668,0.12990,0.18386,0.20673,0.17179,0.17395,0.19751
Fixed 50/50,0.16184,0.12722,0.18247,0.20402,0.16888,0.17113,0.19502
Forward chained global,0.16184,0.12718,0.18220,0.20352,0.16868,0.17090,0.19462
Forward chained present/missing,0.16184,0.12714,0.18247,0.20352,0.16874,0.17096,0.19473
In-sample global w=0.6 (optimistic),0.16185,0.12721,0.18202,0.20352,0.16865,0.17086,0.19461
In-sample present=0.55 missing=0.65 (optimistic),0.16208,0.12733,0.18186,0.20332,0.16865,0.17084,0.19457


Existing weights file preserved: /Users/secilnuragtepe/Desktop/wind-power-project/math482-wind-power-mean-2026/oof/exp04_blend_weights.json
Computed candidate: {'spatial_oof': 'oof/exp03_5seed_oof.csv', 'w_global': 0.6, 'w_present': 0.55, 'w_missing': 0.65}


## 4. Full-train Exp00 model and submission blend

This is a faithful notebook adaptation of the original `submit` mode. It trains the Exp00 3-seed MLP and blends its means with the precomputed spatial submission using the global weight saved from OOF validation. **The spatial submission must come from the matching Exp03/Exp02 pipeline.** No sigma is generated by this experiment. Existing submission files are never overwritten.

In [14]:
from pathlib import Path

weights_path = ROOT / "oof" / "exp04_blend_weights.json"
output_path = ROOT / "exp04_blend_submission.csv"

print("Weights:", weights_path.exists())
print("Existing output:", output_path.exists())

print("\nSpatial submission candidates:")
for p in sorted(ROOT.glob("*submission*.csv")):
    print(p.name)

Weights: True
Existing output: True

Spatial submission candidates:
ensemble_submission.csv
exp02_spatial_submission.csv
exp02_submission_backup.csv
exp03_lrdecay_submission.csv
exp04_blend_submission.csv
final_submission.csv
hgb_submission.csv
nn_submission.csv
p1_grouped_sigma_submission.csv
sample_submission.csv


In [15]:
sp_sub = ROOT / "exp03_lrdecay_submission.csv"
output_path = ROOT / "exp04_blend_reproduced.csv"

print("Spatial file exists:", sp_sub.exists())
print("New output exists:", output_path.exists())

Spatial file exists: True
New output exists: False


In [16]:
RUN_SUBMISSION = True  # Change to True to train and write the CSV
if RUN_SUBMISSION:
    if output_path.exists():
        raise FileExistsError(f"Will not overwrite {output_path}")
    if not weights_path.exists() or not sp_sub.exists():
        raise FileNotFoundError("Need oof/exp04_blend_weights.json and the matching spatial submission")
    from cv_utils import load_data
    from wp_lib import base_features, crossfarm_mean_impute, WindPowerMLP, baseline_matrix, train_eval, NUMERIC
    weights=json.loads(weights_path.read_text())
    train,test,sample=load_data()
    train,test=base_features(train),base_features(test)
    tr_i=crossfarm_mean_impute(train,train)
    te_i=crossfarm_mean_impute(train,test)
    scaler=StandardScaler().fit(tr_i[NUMERIC])
    Xtr,Xte=baseline_matrix(tr_i,scaler),baseline_matrix(te_i,scaler)
    ps=[]
    for seed in (42,43,44):
        p,_,_=train_eval(lambda: WindPowerMLP(Xtr.shape[1]),Xtr,train.power.to_numpy(np.float32),Xte,np.zeros(len(Xte),np.float32),seed=seed)
        ps.append(p)
        print("Exp00 seed",seed,"mean",p.mean())
    exp00=pd.DataFrame(dict(id=test.id.values,e0=np.mean(ps,axis=0),weather_available=test.weather_available.values))
    spatial=pd.read_csv(sp_sub)[["id","mu"]].rename(columns={"mu":"sp"})
    merged=sample[["id"]].merge(exp00,on="id",validate="one_to_one").merge(spatial,on="id",validate="one_to_one")
    w=float(weights["w_global"])
    mu=np.clip(apply_blend(merged,w,w),0,1)
    submission=pd.DataFrame(dict(id=merged.id,mu=mu))
    assert len(submission)==len(sample)==9408
    assert submission.id.equals(sample.id)
    assert np.isfinite(submission.mu).all() and submission.mu.between(0,1).all()
    submission.to_csv(output_path,index=False)
    display(submission.head())
    print("Saved:",output_path,"spatial weight:",w)
else:
    print("Training skipped. Existing files were not modified.")

Exp00 seed 42 mean 0.24157536
Exp00 seed 43 mean 0.238583
Exp00 seed 44 mean 0.2726114


,id,mu
0,245754,0.170012
1,246677,0.040628
2,249992,0.146708
3,250368,0.070491
4,246581,0.024879


Saved: /Users/secilnuragtepe/Desktop/wind-power-project/math482-wind-power-mean-2026/exp04_blend_reproduced.csv spatial weight: 0.6


## 5. Reproducibility notes and limitations

- The ZIP supplied only three scripts. `cv_utils.py`, `wp_lib.py`, original OOF CSVs, the spatial submission CSV, and training CSVs were not supplied, so this notebook **cannot be executed end-to-end or certified to reproduce the original submission here**.
- The script chooses Exp03 spatial predictions when `oof/exp03_5seed_oof.csv` exists, otherwise Exp02. This choice must match the original run.
- Exp04 chooses the final global weight from all OOF rows. This is appropriate for training the final blend but its in-sample validation score is optimistic; use the forward-chained score to assess generalization.
- The output has two columns (`id,mu`) because the original source code does. A three-column probabilistic competition submission would need a separately justified uncertainty model.
- Exact numeric reproduction can depend on the training environment, PyTorch version and device.
- The original `exp03_lrdecay_cv.py` can regenerate the spatial submission but requires additional local modules, full training data, and model training.


In [18]:
import pandas as pd
import json

original = pd.read_csv(ROOT / "exp04_blend_submission.csv")
spatial = pd.read_csv(ROOT / "exp03_lrdecay_submission.csv")

with open(ROOT / "oof" / "exp04_blend_weights.json") as f:
    weights = json.load(f)

print("Original CSV:", original.shape)
print("Spatial CSV:", spatial.shape)
print("Original columns:", original.columns.tolist())
print("Spatial columns:", spatial.columns.tolist())
print("Saved weights:", weights)
print("Original IDs unique:", original["id"].is_unique)
print("Spatial IDs unique:", spatial["id"].is_unique)

Original CSV: (9408, 2)
Spatial CSV: (9408, 2)
Original columns: ['id', 'mu']
Spatial columns: ['id', 'mu']
Saved weights: {'spatial_oof': 'oof/exp03_5seed_oof.csv', 'w_global': 0.6, 'w_present': 0.55, 'w_missing': 0.65}
Original IDs unique: True
Spatial IDs unique: True


In [30]:
import pandas as pd
import numpy as np

old = pd.read_csv(ROOT / "exp04_original_backup.csv")
new = pd.read_csv(ROOT / "exp04_blend_reproduced.csv")

assert len(old) == len(new) == 9408
assert old["id"].is_unique and new["id"].is_unique
assert set(old["id"]) == set(new["id"])

merged = old.merge(new, on="id", suffixes=("_old", "_new"))

diff = np.abs(merged["mu_old"] - merged["mu_new"])

print("Compared rows:", len(merged))
print("Maximum difference:", diff.max())
print("Mean difference:", diff.mean())
print("Exactly equal:", bool((diff == 0).all()))

Compared rows: 9408
Maximum difference: 0.0
Mean difference: 0.0
Exactly equal: True


In [32]:
from pathlib import Path

print("P2 reproduced exists:", Path("p2_D_joint_reproduced.csv").exists())

P2 reproduced exists: True
